# Session 09: MCP Servers

## Understanding and Building with the Model Context Protocol

### Learning Objectives

- **Understand the Model Context Protocol (MCP)** — what it is, why it matters, and how it enables AI models to interact with external tools and data sources
- **Consume remote MCP servers** — connect to public MCP servers (GitMCP, GitHub) and use their tools through the Anthropic-powered agent
- **Build a custom MCP server** — create a Stone Ridge investment tools server using `FastMCP`
- **Integrate MCP servers with LangGraph** — use `langchain-mcp-adapters` and `MultiServerMCPClient` to bring MCP tools into a LangGraph agent

### Overview

The **Model Context Protocol (MCP)** is an open standard that defines how AI models connect to external tools and data sources. Think of it as a universal adapter:

- **Without MCP**: Every AI app needs custom integration code for every tool/API
- **With MCP**: Tools expose themselves via a standard protocol; any MCP-compatible client can discover and use them

In this notebook we will:
1. Connect to **remote MCP servers** (GitMCP for documentation, GitHub MCP for repo operations)
2. Build a **custom MCP server** (`mcp_server.py`) with Stone Ridge investment tools
3. Consume our custom server from a **LangGraph agent** via `langchain-mcp-adapters`

---

## Task 1: Dependencies & Environment

In [1]:
import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key:")

# Optional: GitHub PAT for GitHub MCP Server
if not os.environ.get("GITHUB_PAT"):
    pat = getpass.getpass("GitHub PAT (optional — press Enter to skip):")
    if pat.strip():
        os.environ["GITHUB_PAT"] = pat

In [2]:
import nest_asyncio
nest_asyncio.apply()  # Required for async MCP operations in Jupyter

In [3]:
from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-sonnet-4-20250514", temperature=0)

# Test the connection
response = llm.invoke("Say 'MCP server ready!' in exactly those words.")
print(response.content)

OMP: Warning #179: Function Can't set size of /tmp file failed:


MCP server ready!


---

## Task 2: Remote MCP Servers — GitMCP

**GitMCP** turns any GitHub repository's documentation into a searchable MCP server — no authentication required for public repos.

We'll connect to the LangChain documentation via GitMCP to demonstrate the protocol in action.

In [4]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# Connect to GitMCP — a public MCP server for GitHub documentation
gitmcp_client = MultiServerMCPClient(
    {
        "langchain_docs": {
            "transport": "http",
            "url": "https://gitmcp.io/langchain-ai/langchain",
        }
    }
)

gitmcp_tools = await gitmcp_client.get_tools()

print(f"Loaded {len(gitmcp_tools)} tools from GitMCP:")
for t in gitmcp_tools:
    print(f"  - {t.name}: {t.description[:80]}...")

Unknown SSE event: endpoint


Loaded 4 tools from GitMCP:
  - fetch_langchain_documentation: Fetch entire documentation file from GitHub repository: langchain-ai/langchain. ...
  - search_langchain_documentation: Semantically search within the fetched documentation from GitHub repository: lan...
  - search_langchain_code: Search for code within the GitHub repository: "langchain-ai/langchain" using the...
  - fetch_generic_url_content: Generic tool to fetch content from any absolute URL, respecting robots.txt rules...


### Using GitMCP tools in a LangGraph Agent

Let's wire these tools into a simple ReAct agent and ask a documentation question.

In [5]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, SystemMessage


class AgentState(TypedDict):
    messages: Annotated[list, add_messages]


llm_with_gitmcp = llm.bind_tools(gitmcp_tools)


def agent_node(state: AgentState):
    messages = [
        SystemMessage(content="You are a helpful assistant with access to documentation tools. Use them to answer questions accurately.")
    ] + state["messages"]
    return {"messages": [llm_with_gitmcp.invoke(messages)]}


def should_continue(state: AgentState):
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "tools"
    return "end"


workflow = StateGraph(AgentState)
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(gitmcp_tools, handle_tool_errors=True))
workflow.add_edge(START, "agent")
workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
workflow.add_edge("tools", "agent")

gitmcp_agent = workflow.compile()
print("GitMCP agent compiled!")

GitMCP agent compiled!


In [6]:
result = gitmcp_agent.invoke(
    {"messages": [HumanMessage(content="How do I create a custom tool in LangChain?")]}
)
print(result["messages"][-1].content[:1500])

I apologize for the technical issues with the documentation tools. Let me provide you with comprehensive information about creating custom tools in LangChain based on my knowledge:

## Creating Custom Tools in LangChain

There are several ways to create custom tools in LangChain. Here are the main approaches:

### 1. Using the `@tool` Decorator (Simplest Method)

```python
from langchain.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together."""
    return a * b

# Usage
result = multiply.invoke({"a": 3, "b": 4})
print(result)  # 12
```

### 2. Using the `Tool` Class

```python
from langchain.tools import Tool

def search_function(query: str) -> str:
    """Custom search function"""
    # Your custom logic here
    return f"Search results for: {query}"

custom_tool = Tool(
    name="CustomSearch",
    description="A custom search tool that searches for information",
    func=search_function
)
```

### 3. Using `StructuredTool` for Complex Inp

---

## Task 3: Remote MCP Servers — GitHub MCP

The **GitHub MCP Server** is an official, GitHub-maintained server that gives agents the ability to manage repositories, issues, pull requests, and more.

> **Requires**: A GitHub Personal Access Token (fine-grained) with `Contents: Read and write`, `Pull requests: Read and write`, and `Metadata: Read-only` permissions.

| MCP Tool | Replaces | What It Does |
|---|---|---|
| `create_repository` | `git init` + GitHub UI | Creates a new repo |
| `create_or_update_file` | `git add` + `git commit` | Commits a file to a branch |
| `create_branch` | `git checkout -b` | Creates a new branch |
| `create_pull_request` | `gh pr create` | Opens a PR |
| `search_repositories` | `gh repo list` | Searches repos |
| `get_file_contents` | `git show` / `cat` | Reads a file |
| `list_commits` | `git log` | Shows commit history |

In [7]:
if os.environ.get("GITHUB_PAT"):
    github_client = MultiServerMCPClient(
        {
            "github": {
                "transport": "http",
                "url": "https://api.githubcopilot.com/mcp/",
                "headers": {
                    "Authorization": f"Bearer {os.environ['GITHUB_PAT']}",
                },
            }
        }
    )

    github_mcp_tools = await github_client.get_tools()
    print(f"Loaded {len(github_mcp_tools)} GitHub MCP tools:")
    for t in github_mcp_tools:
        print(f"  - {t.name}: {t.description[:80]}...")
else:
    print("GITHUB_PAT not set — skipping GitHub MCP. Set it above to enable.")
    github_mcp_tools = []

Loaded 44 GitHub MCP tools:
  - add_comment_to_pending_review: Add review comment to the requester's latest pending pull request review. A pend...
  - add_issue_comment: Add a comment to a specific issue in a GitHub repository. Use this tool to add c...
  - add_reply_to_pull_request_comment: Add a reply to an existing pull request comment. This creates a new comment that...
  - assign_copilot_to_issue: Assign Copilot to a specific issue in a GitHub repository.

This tool can help w...
  - create_branch: Create a new branch in a GitHub repository...
  - create_or_update_file: Create or update a single file in a GitHub repository. 
If updating, you should ...
  - create_pull_request: Create a new pull request in a GitHub repository....
  - create_pull_request_with_copilot: Delegate a task to GitHub Copilot coding agent to perform in the background. The...
  - create_repository: Create a new GitHub repository in your account or specified organization...
  - delete_file: Delete a file from

---

## Task 4: Build a Custom MCP Server

Now let's look at **building our own MCP server**. The file `mcp_server.py` in this directory implements a Stone Ridge investment tools server using `FastMCP`.

### Server Architecture

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Stone Ridge Investment Tools")

@mcp.tool()
def get_fund_overview(fund_name: str) -> str:
    """Return an overview of a Stone Ridge fund."""
    ...

@mcp.tool()
def get_investment_philosophy() -> str:
    """Return Stone Ridge's investment philosophy."""
    ...

@mcp.tool()
def analyze_portfolio_allocation(...) -> str:
    """Analyze a portfolio from a Stone Ridge perspective."""
    ...

@mcp.tool()
def search_investor_letter(query: str, section: str = "all") -> str:
    """Keyword search across the investor letter."""
    ...

if __name__ == "__main__":
    mcp.run(transport="stdio")
```

### Testing with MCP Inspector

You can test the server interactively:

```bash
uv run mcp dev mcp_server.py
```

This launches the **MCP Inspector** — a web UI where you can call each tool and see the responses.

Let's review the server code:

In [8]:
# Let's peek at the MCP server code
with open("mcp_server.py") as f:
    print(f.read())

"""Stone Ridge Investment MCP Server — example custom MCP server.

Provides four investment-themed tools with hardcoded data derived from
the Stone Ridge 2025 Investor Letter.

Run with:
    uv run mcp dev mcp_server.py
"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("Stone Ridge Investment Tools")

# ---------------------------------------------------------------------------
# Hardcoded reference data (sourced from the 2025 Investor Letter)
# ---------------------------------------------------------------------------

FUNDS = {
    "sre": {
        "name": "Stone Ridge Reinsurance Risk Premium Interval Fund (SRE)",
        "overview": (
            "SRE provides investors with access to the reinsurance risk premium — "
            "the return for bearing catastrophe risk such as hurricanes and earthquakes. "
            "Stone Ridge believes reinsurance offers a genuinely uncorrelated return "
            "stream driven by physical events rather than financial-market sentime

### Quick direct test of the MCP server functions

Since the tools are regular Python functions, we can import and call them directly:

In [9]:
# Direct import test (no MCP transport needed)
from mcp_server import get_fund_overview, get_investment_philosophy, analyze_portfolio_allocation, search_investor_letter

print("=== Fund Overview: Bitcoin ===")
print(get_fund_overview(fund_name="bitcoin"))
print()

print("=== Portfolio Analysis ===")
print(analyze_portfolio_allocation(
    equities_pct=50.0,
    fixed_income_pct=25.0,
    alternatives_pct=20.0,
    cash_pct=5.0,
))
print()

print("=== Letter Search: 'Bayesian' ===")
print(search_investor_letter(query="Bayesian"))

=== Fund Overview: Bitcoin ===
**Stone Ridge Bitcoin Strategy**

Stone Ridge allocates a portion of its balance sheet to bitcoin, viewing it as a long-duration, non-sovereign store of value. The 2025 letter discusses bitcoin's role in portfolio construction through the lens of Bayesian probability updating and convexity.

Key themes:
  - Non-sovereign store of value thesis
  - Bayesian probability of monetary adoption
  - Convex payoff profile

=== Portfolio Analysis ===
## Portfolio Allocation Analysis (Stone Ridge Perspective)

| Asset Class    | Allocation |
|----------------|------------|
| Equities       | 50.0%      |
| Fixed Income   | 25.0%      |
| Alternatives   | 20.0%      |
| Cash           | 5.0%      |

### Stone Ridge Commentary

- **Moderate alternatives allocation**: Reasonable starting point.  Consider whether your alternatives are truly independent of equity-market beta — many hedge-fund strategies retain hidden equity correlation.

=== Letter Search: 'Bayesian' ===

---

## Task 5: Consume the Custom MCP Server with `langchain-mcp-adapters`

Now we'll connect to our custom MCP server using `MultiServerMCPClient` and use it inside a LangGraph agent. This demonstrates the full MCP lifecycle:

1. **Build** a server (`mcp_server.py`)
2. **Connect** via `langchain-mcp-adapters`
3. **Use** the tools in a LangGraph agent

The `MultiServerMCPClient` supports `stdio` transport — it launches the server as a subprocess and communicates over stdin/stdout.

In [10]:
import sys

# Connect to our custom MCP server via stdio transport
custom_mcp_client = MultiServerMCPClient(
    {
        "stone_ridge": {
            "transport": "stdio",
            "command": sys.executable,  # Use the current Python interpreter
            "args": ["mcp_server.py"],
        }
    }
)

stone_ridge_tools = await custom_mcp_client.get_tools()

print(f"Loaded {len(stone_ridge_tools)} tools from Stone Ridge MCP server:")
for t in stone_ridge_tools:
    print(f"  - {t.name}: {t.description[:80]}...")

Loaded 4 tools from Stone Ridge MCP server:
  - get_fund_overview: Return an overview of a Stone Ridge fund.

Args:
    fund_name: Name or keyword ...
  - get_investment_philosophy: Return Stone Ridge's investment philosophy as described in the 2025 Investor Let...
  - analyze_portfolio_allocation: Analyze a portfolio allocation from a Stone Ridge perspective.

The analysis use...
  - search_investor_letter: Keyword search across sections of the Stone Ridge 2025 Investor Letter.

Args:
 ...


In [11]:
# Build a LangGraph agent with our custom MCP tools
llm_with_sr = llm.bind_tools(stone_ridge_tools)


def sr_agent_node(state: AgentState):
    messages = [
        SystemMessage(
            content="You are the Stone Ridge Investment Assistant. Use your tools to answer "
            "questions about Stone Ridge funds, investment philosophy, and portfolio analysis."
        )
    ] + state["messages"]
    return {"messages": [llm_with_sr.invoke(messages)]}


sr_workflow = StateGraph(AgentState)
sr_workflow.add_node("agent", sr_agent_node)
sr_workflow.add_node("tools", ToolNode(stone_ridge_tools, handle_tool_errors=True))
sr_workflow.add_edge(START, "agent")
sr_workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
sr_workflow.add_edge("tools", "agent")

sr_agent = sr_workflow.compile()
print("Stone Ridge MCP agent compiled!")

Stone Ridge MCP agent compiled!


In [12]:
result = sr_agent.invoke(
    {"messages": [HumanMessage(content="What is Stone Ridge's investment philosophy? How do they think about bitcoin?")]}
)
print(result["messages"][-1].content)

[03/24/26 01:12:29] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=15234;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=106586;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/24/26 01:12:31] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=619502;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=357294;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/24/26 01:12:38] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=528096;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=484651;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

I apologize, but I'm experiencing technical difficulties accessing the Stone Ridge information tools at the moment. However, I can provide you with some general context about Stone Ridge's approach based on what I know:

**Stone Ridge's Investment Philosophy** typically centers around:

1. **Bayesian Diversification** - They focus on building portfolios with truly independent risk factors rather than traditional asset class diversification
2. **Tail Risk Harvesting** - Seeking opportunities in areas where others fear to tread, particularly in tail risk scenarios
3. **Long-term Value Creation** - Taking contrarian positions with long investment horizons
4. **Risk-adjusted Returns** - Focusing on absolute returns rather than relative performance

**Regarding Bitcoin**, Stone Ridge has been notably bullish and was one of the early institutional adopters. They generally view Bitcoin as:

- A store of value and potential hedge against monetary debasement
- An uncorrelated asset that provide

In [13]:
result = sr_agent.invoke(
    {"messages": [HumanMessage(
        content="Analyze this portfolio: 40% equities, 30% fixed income, 25% alternatives, 5% cash. "
        "What would Stone Ridge say about this allocation?"
    )]}
)
print(result["messages"][-1].content)

[03/24/26 01:12:40] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=178320;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=196429;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/24/26 01:12:42] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=845827;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=312081;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/24/26 01:12:44] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=584517;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=785477;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/24/26 01:12:53] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=483482;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=554159;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

I apologize, but I'm experiencing technical difficulties with the analysis tools. However, I can provide you with Stone Ridge's likely perspective on your portfolio allocation based on their known investment philosophy:

**Your Portfolio: 40% Equities, 30% Fixed Income, 25% Alternatives, 5% Cash**

From Stone Ridge's perspective, this allocation would likely be viewed favorably in several key areas:

**Strengths:**
1. **Meaningful Alternatives Allocation (25%)** - Stone Ridge strongly advocates for alternatives as they provide access to independent risk factors and potential tail-risk harvesting opportunities
2. **Diversification Beyond Traditional Assets** - The 70% traditional/30% alternative split shows recognition that diversification requires more than just stocks and bonds
3. **Balanced Approach** - The allocation doesn't over-concentrate in any single asset class

**Potential Stone Ridge Observations:**
1. **Quality of Alternatives Matters** - They would emphasize that the 25% a

---

## Task 6: Multi-Server MCP Agent

The real power of MCP is combining **multiple servers** into a single agent. Let's build an agent that has access to:
- **Stone Ridge tools** (our custom server)
- **GitMCP tools** (documentation search)
- **GitHub MCP tools** (if PAT configured)

In [14]:
# Combine all available MCP tools
all_mcp_tools = stone_ridge_tools + gitmcp_tools
if github_mcp_tools:
    all_mcp_tools += github_mcp_tools

print(f"Total MCP tools available: {len(all_mcp_tools)}")

llm_multi = llm.bind_tools(all_mcp_tools)


def multi_agent_node(state: AgentState):
    messages = [
        SystemMessage(
            content="You are an investment research assistant with access to Stone Ridge investment tools, "
            "documentation search, and GitHub repository tools. Use the appropriate tools for each query."
        )
    ] + state["messages"]
    return {"messages": [llm_multi.invoke(messages)]}


multi_wf = StateGraph(AgentState)
multi_wf.add_node("agent", multi_agent_node)
multi_wf.add_node("tools", ToolNode(all_mcp_tools, handle_tool_errors=True))
multi_wf.add_edge(START, "agent")
multi_wf.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
multi_wf.add_edge("tools", "agent")

multi_agent = multi_wf.compile()
print("Multi-server MCP agent compiled!")

Total MCP tools available: 52
Multi-server MCP agent compiled!


In [15]:
result = multi_agent.invoke(
    {"messages": [HumanMessage(
        content="Search the Stone Ridge investor letter for information about reinsurance, "
        "then look up how LangChain implements tool calling."
    )]}
)
print(result["messages"][-1].content[:2000])

[03/24/26 01:12:56] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=382815;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=111491;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/24/26 01:12:58] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=129274;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=617643;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/24/26 01:13:00] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=153401;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=429904;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 ]8;id=2537;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=202191;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             Too Many Requests"                                                                    

                    INFO     Retrying request to /v1/messages in 16.000000 seconds             ]8;id=876629;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/anthropic/_base_client.py\_base_client.py]8;;\:]8;id=258180;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/anthropic/_base_client.py#1161\1161]8;;\

...

[03/24/26 01:14:06] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=867237;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=38966;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 ]8;id=647127;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=317370;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             Too Many Requests"                                                                    

                    INFO     Retrying request to /v1/messages in 23.000000 seconds             ]8;id=441130;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/anthropic/_base_client.py\_base_client.py]8;;\:]8;id=728805;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/anthropic/_base_client.py#1161\1161]8;;\

[03/24/26 01:14:35] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=996317;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=494153;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

I apologize, but I'm experiencing technical issues with the available tools at the moment. The tools are returning "NotImplementedError" indicating they don't support synchronous invocation. 

To help you with your request:

1. **For Stone Ridge reinsurance information**: I would normally search their investor letter for details about their reinsurance strategy, risk management approach, and performance metrics using their specialized search tools.

2. **For LangChain tool calling implementation**: I would typically search their documentation and codebase to find information about how they implement tool calling functionality, including their agent frameworks, tool interfaces, and execution patterns.

Could you try your request again in a moment, or would you like me to attempt a different approach to gather this information? The technical issue appears to be temporary with the tool execution system.


### Question #1

What are the benefits of the MCP protocol over writing custom API wrappers? When would you still prefer a direct `@tool` wrapper instead?

##### Answer

*MCP protocol is more flexible and open-ended, whereas custom API wrappers give you more precise and fine-grained control over the output you expect to receieve. If I'm running precise financal calculations or expect my data to be in a very specific format, I would probably prefer user a @tool wrapper instead of an MCP*

### Activity #1

Extend the `mcp_server.py` with a new tool:

1. Add a `compare_funds(fund_a: str, fund_b: str)` tool that returns a side-by-side comparison
2. Restart the MCP server connection
3. Test it through the agent by asking: *"Compare the SRE fund with the Bitcoin strategy"*

In [16]:
# ---------------------------------
# Redfine MCP Server
# ----------------------------------
custom_mcp_client = MultiServerMCPClient(
    {
        "stone_ridge": {
            "transport": "stdio",
            "command": sys.executable,  # Use the current Python interpreter
            "args": ["mcp_server.py"],
        }
    }
)

stone_ridge_tools = await custom_mcp_client.get_tools()

# --------------------------------
# Build new graph with new tools
# --------------------------------
llm_with_sr = llm.bind_tools(stone_ridge_tools)


def sr_agent_node(state: AgentState):
    messages = [
        SystemMessage(
            content="You are the Stone Ridge Investment Assistant. Use your tools to answer "
            "questions about Stone Ridge funds, investment philosophy, and portfolio analysis."
        )
    ] + state["messages"]
    return {"messages": [llm_with_sr.invoke(messages)]}

sr_workflow = StateGraph(AgentState)
sr_workflow.add_node("agent", sr_agent_node)
sr_workflow.add_node("tools", ToolNode(stone_ridge_tools, handle_tool_errors=True))
sr_workflow.add_edge(START, "agent")
sr_workflow.add_conditional_edges("agent", should_continue, {"tools": "tools", "end": END})
sr_workflow.add_edge("tools", "agent")

sr_agent = sr_workflow.compile()

# --------------------------------
# Invoke new graph with question
# --------------------------------
result = multi_agent.invoke(
    {"messages": [HumanMessage(
        content="Compare the SRE fund with the Bitcoin strategy"
    )]}
)
print(result["messages"][-1].content[:2000])

[03/24/26 01:23:52] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=863682;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=291757;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/24/26 01:23:55] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=255934;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=943113;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

[03/24/26 01:23:57] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=834748;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=501482;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

                    INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 429 ]8;id=360430;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=24176;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             Too Many Requests"                                                                    

                    INFO     Retrying request to /v1/messages in 16.000000 seconds             ]8;id=417467;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/anthropic/_base_client.py\_base_client.py]8;;\:]8;id=345446;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/anthropic/_base_client.py#1161\1161]8;;\

[03/24/26 01:24:21] INFO     HTTP Request: POST https://api.anthropic.com/v1/messages "HTTP/1.1 200 ]8;id=660096;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=435952;file:///Users/alexei.naumann/Desktop/AIE/AI-Engineering/09_Production_and_MCP/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

I'm experiencing technical difficulties with the Stone Ridge investment tools at the moment. However, I can provide you with a general framework for comparing these two strategies based on typical characteristics of reinsurance and Bitcoin investments:

## Key Comparison Areas:

### **Risk Profile**
- **SRE (Reinsurance)**: Typically involves catastrophe risk, underwriting cycles, and regulatory risks
- **Bitcoin**: High volatility, regulatory uncertainty, and technology/adoption risks

### **Return Characteristics**
- **SRE**: Generally aims for steady, insurance-like returns with occasional large losses from catastrophic events
- **Bitcoin**: Historically high volatility with potential for significant gains or losses

### **Correlation with Traditional Assets**
- **SRE**: Often low correlation with equity markets, providing diversification benefits
- **Bitcoin**: Correlation with traditional assets has varied over time

### **Investment Horizon**
- **SRE**: Typically medium to long-t

---

## Summary

In this notebook we covered:

- **MCP Protocol Concepts**: How the Model Context Protocol enables standardized tool discovery and invocation
- **Remote MCP Servers**: Connected to GitMCP (documentation) and GitHub MCP (repo operations)
- **Custom MCP Server**: Built `mcp_server.py` with Stone Ridge investment tools using `FastMCP`
- **langchain-mcp-adapters**: Used `MultiServerMCPClient` to bridge MCP tools into LangGraph agents
- **Multi-Server Agent**: Combined tools from multiple MCP servers into a single agent

### Key Takeaways

1. **MCP is a protocol, not a product** — any server that implements MCP can be consumed by any MCP-compatible client
2. **`FastMCP` makes server creation trivial** — just decorate Python functions with `@mcp.tool()`
3. **`langchain-mcp-adapters` bridges MCP → LangChain** — MCP tools become regular LangChain tools
4. **Multiple servers compose naturally** — an agent can use tools from many servers simultaneously